# The Block Trade Decision Agent

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python (`pandas`, `scikit-learn`, `matplotlib`), familiarity with equity capital markets terminology, and comfort reading deal-style metrics.

            **What You Will Learn:**

            - Describe how ownership clarity, deal terms, and passive-flow dynamics affect a block-trade participation decision.
- Compare a simple rules-based screen against a supervised classification model for time-sensitive investment decisions.
- Adapt a three-agent decision pattern to another capital-markets workflow that depends on fast context fusion.

            **Estimated time:** 45 minutes

            **Collection context:** Agentic notebooks for block trades, media monitoring, cyber triage, market integrity, legal strategy, and internal crossing decisions.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** A block trade can require a participation decision in under an hour, and a weak read on ownership, supply pressure, or index demand can cost millions in slippage.

**Operational question:** Should the investment team participate in this large secondary share sale at the proposed terms?

**Primary stakeholders:** equity capital markets teams, portfolio managers, sales traders, and investment committees

**Decision supported:** Combine fast ownership work, deal parsing, and index-flow estimates into a documented participation recommendation.

**Comprehension check:** Before looking at the data, write one sentence describing why a false positive could be more expensive than a false negative in a crowded block.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Use a lightweight stack with deterministic synthetic data so the workflow remains runnable in Colab or local Jupyter.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
N_DEALS = 1200

print("Environment ready for a synthetic block-trade workflow.")


## Step 3 — Data Creation & Context

We simulate candidate block trades with ownership confidence, discount terms, supply pressure, and estimated passive index demand to mimic the speed of real placement decisions.


In [ ]:
deal_df = pd.DataFrame({
    "beneficial_owner_confidence": RNG.uniform(0.35, 0.98, N_DEALS),
    "discount_to_close_pct": RNG.uniform(0.5, 8.0, N_DEALS),
    "book_coverage_ratio": RNG.uniform(0.6, 3.4, N_DEALS),
    "index_effect_bps": RNG.normal(12, 34, N_DEALS).clip(-75, 95),
    "free_float_unlock_pct": RNG.uniform(0.0, 14.0, N_DEALS),
    "seller_overhang_score": RNG.uniform(0.0, 10.0, N_DEALS),
    "liquidity_days": RNG.uniform(1.5, 38.0, N_DEALS),
})

latent_signal = (
    0.95 * deal_df["discount_to_close_pct"]
    + 1.10 * deal_df["book_coverage_ratio"]
    + 0.028 * deal_df["index_effect_bps"]
    + 0.12 * deal_df["liquidity_days"]
    - 1.25 * deal_df["seller_overhang_score"]
    - 0.55 * deal_df["free_float_unlock_pct"]
    - 5.50 * (1 - deal_df["beneficial_owner_confidence"])
    + RNG.normal(0, 1.8, N_DEALS)
)
threshold = np.quantile(latent_signal, 0.52)
deal_df["participation_decision"] = np.where(
    latent_signal >= threshold, "Participate", "Pass"
)

print("Synthetic deal sample:")
print(deal_df.head(3).round(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Inspect how discount and estimated index support interact before building the model. The goal is to connect market structure intuition to the classification task.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.scatterplot(
    data=deal_df,
    x="discount_to_close_pct",
    y="index_effect_bps",
    hue="participation_decision",
    alpha=0.7,
    ax=axes[0],
)
axes[0].set_title("Block Trade: Discount vs. Index Effect")
axes[0].set_xlabel("Discount to Close (%)")
axes[0].set_ylabel("Estimated Index Effect (bps)")

sns.histplot(
    data=deal_df,
    x="seller_overhang_score",
    hue="participation_decision",
    bins=24,
    multiple="stack",
    ax=axes[1],
)
axes[1].set_title("Block Trade: Overhang Distribution")
axes[1].set_xlabel("Seller Overhang Score")

plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The EDA contrasts how attractive discount and passive buying support lift participation odds while heavy seller overhang pushes similar deals toward a pass decision.


## Step 5 — Feature Engineering

We add interpretable composite features that mimic specialized agents: ownership risk, execution support, and supply pressure.


In [ ]:
analysis_df = deal_df.copy()
analysis_df["owner_risk_gap"] = 1 - analysis_df["beneficial_owner_confidence"]
analysis_df["execution_support_score"] = (
    analysis_df["discount_to_close_pct"]
    + analysis_df["book_coverage_ratio"]
    + analysis_df["index_effect_bps"] / 40
)
analysis_df["supply_pressure_score"] = (
    analysis_df["seller_overhang_score"] + analysis_df["free_float_unlock_pct"] / 2
)

feature_columns = [
    "beneficial_owner_confidence",
    "discount_to_close_pct",
    "book_coverage_ratio",
    "index_effect_bps",
    "free_float_unlock_pct",
    "seller_overhang_score",
    "liquidity_days",
    "owner_risk_gap",
    "execution_support_score",
    "supply_pressure_score",
]

print(analysis_df[feature_columns].head(3).round(3).to_string(index=False))


## Step 6 — Baseline Establishment

A simple desk heuristic is a reasonable baseline: participate only when discount is wide enough and index support is positive.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    analysis_df[feature_columns],
    analysis_df["participation_decision"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["participation_decision"],
)

baseline_pred = np.where(
    (X_test["discount_to_close_pct"] >= 4.0)
    & (X_test["index_effect_bps"] >= 0),
    "Participate",
    "Pass",
)

print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")
print(f"Baseline F1: {f1_score(y_test, baseline_pred, pos_label='Participate'):.3f}")


## Step 7 — Model Training & Evaluation

Train a stronger classifier and compare it directly with the desk heuristic. Prediction prompt: which feature should matter most, and why?


In [ ]:
participation_model = RandomForestClassifier(
    n_estimators=260,
    min_samples_leaf=5,
    random_state=RANDOM_SEED,
)
participation_model.fit(X_train, y_train)
model_pred = participation_model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, model_pred):.3f}")
print(f"Model F1: {f1_score(y_test, model_pred, pos_label='Participate'):.3f}")
print(classification_report(y_test, model_pred))

conf = confusion_matrix(y_test, model_pred, labels=["Pass", "Participate"])
sns.heatmap(
    conf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pass", "Participate"],
    yticklabels=["Pass", "Participate"],
)
plt.title("Participation Recommendation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()
plt.close()


## Step 8 — Interpretability & Explainability

Feature importance lets the team see whether the model is emphasizing the same drivers that a senior trader would expect.


In [ ]:
importance_frame = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": participation_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

sns.barplot(data=importance_frame, y="feature", x="importance", color="#2A9D8F")
plt.title("Most Important Drivers of Block Participation")
plt.xlabel("Model Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
plt.close()

print(importance_frame.head(6).round(3).to_string(index=False))


## Step 9 — Operational Application

Use the model as the final stage of a three-agent workflow: ownership agent, deal-terms agent, and index-effect agent.


In [ ]:
scenario_df = pd.DataFrame(
    {
        "beneficial_owner_confidence": [0.92, 0.58, 0.71],
        "discount_to_close_pct": [3.8, 6.9, 2.6],
        "book_coverage_ratio": [2.6, 1.2, 3.1],
        "index_effect_bps": [22, -18, 58],
        "free_float_unlock_pct": [2.0, 11.5, 4.0],
        "seller_overhang_score": [2.5, 8.4, 3.6],
        "liquidity_days": [24, 8, 33],
    },
    index=["quality_secondary", "messy_exit", "index_tailwind"],
)
scenario_df["owner_risk_gap"] = 1 - scenario_df["beneficial_owner_confidence"]
scenario_df["execution_support_score"] = (
    scenario_df["discount_to_close_pct"]
    + scenario_df["book_coverage_ratio"]
    + scenario_df["index_effect_bps"] / 40
)
scenario_df["supply_pressure_score"] = (
    scenario_df["seller_overhang_score"] + scenario_df["free_float_unlock_pct"] / 2
)

scenario_prob = participation_model.predict_proba(scenario_df[feature_columns])
participate_index = list(participation_model.classes_).index("Participate")

scenario_df["ownership_agent_view"] = np.where(
    scenario_df["beneficial_owner_confidence"] >= 0.8, "clean_owner", "needs_more_work"
)
scenario_df["deal_parser_agent_view"] = np.where(
    scenario_df["discount_to_close_pct"] >= 4.0, "wide_discount", "tight_terms"
)
scenario_df["index_effect_agent_view"] = np.where(
    scenario_df["index_effect_bps"] >= 10, "forced_buying_support", "limited_support"
)
scenario_df["participate_probability"] = scenario_prob[:, participate_index]
scenario_df["final_recommendation"] = np.where(
    scenario_df["participate_probability"] >= 0.55, "Participate", "Pass"
)

print(scenario_df.round(3).to_string())


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic block-trade decision workflow that fuses ownership research, deal-term extraction, and index-flow logic into a documented participation recommendation.

            **Limitations to note:**

            - The dataset is synthetic and compresses legal, settlement, and liquidity nuances into a small set of features.
- Execution risk, counterparty relationships, and live order-book conditions are simplified.
- The model is a teaching aid and should not be used for real investment decisions without validated market data.

            **Ethical reflection:** A fast participation recommendation can amplify market pressure if the model embeds stale assumptions about liquidity, ownership, or index behavior, so human review remains essential.

            **Reflection questions:**

            1. Which missing live-market signal would you add first before trusting this workflow on a real block?
2. How would the cost of a false positive change for a thinly traded stock versus a mega-cap constituent?
3. How could you adapt the three-agent design to a convertible offering or accelerated bookbuild?


            ## References

            - Breiman, L. (2001). Random Forests.
- Scenario inspiration: equity capital markets block-trade workflows and passive index-flow analysis described in public market microstructure literature.
- Scikit-learn user guide: tree-based classification and model evaluation.
